# 🛡️ Mini SOC — GRPO Reinforcement Learning Training

**Repository:** [github.com/blackcoderh4k/mini-soc](https://github.com/blackcoderh4k/mini-soc)

This notebook trains a `Qwen2.5-0.5B-Instruct` (or 1.5B) language model to act as an autonomous
Tier-1 SOC analyst using **Group Relative Policy Optimization (GRPO)**.

### What this notebook does:
1. Clones the Mini SOC repository
2. Installs all dependencies
3. Starts the FastAPI environment server in the background
4. Runs GRPO training for 200 steps
5. Merges LoRA weights into the final model
6. Plots the reward curves
7. Runs the final benchmark test

### Hardware:
- **Recommended:** Colab T4 (16GB) — runs the 1.5B model comfortably
- **Minimum:** Colab T4 — can run 0.5B model

---
**Estimated time:** ~60–90 minutes for 200 steps on T4

## Step 1: Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
print('GPU is ready!' if result.returncode == 0 else 'WARNING: No GPU detected. Go to Runtime > Change Runtime Type > T4 GPU')

## Step 2: Clone the Repository

In [ ]:
import os

REPO_URL = 'https://github.com/blackcoderh4k/mini-soc.git'
REPO_DIR = '/content/mini-soc'

if os.path.exists(REPO_DIR):
    print('Repo already cloned. Pulling latest...')
    os.chdir(REPO_DIR)
    os.system('git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

print(f'Working directory: {os.getcwd()}')
os.system('ls -la')

## Step 3: Install Dependencies

In [ ]:
# Install all required packages
os.system('pip install -q fastapi uvicorn pydantic httpx openai pyyaml')
os.system('pip install -q transformers>=4.40.0 peft>=0.10.0 accelerate>=0.29.0')
os.system('pip install -q trl>=0.15.0 datasets>=2.19.0')
os.system('pip install -q bitsandbytes>=0.43.0')
os.system('pip install -q matplotlib numpy requests')
print('All packages installed!')

## Step 4: Start the Mini SOC Environment Server

In [ ]:
import subprocess, time, requests

os.chdir(REPO_DIR)

# Start the FastAPI server in the background
server_process = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print('Waiting for server to start...')
for i in range(15):
    time.sleep(2)
    try:
        r = requests.get('http://localhost:8000/health', timeout=3)
        if r.status_code == 200:
            print(f'Server is UP! Health: {r.json()}')
            break
    except:
        print(f'  Attempt {i+1}/15 — waiting...')
else:
    print('WARNING: Server may not have started. Check server logs.')
    stderr = server_process.stderr.read(500)
    print(f'Server stderr: {stderr}')

## Step 5: Verify Environment (Quick Test)

In [ ]:
import json

# Test reset
r = requests.post('http://localhost:8000/reset', json={'task_id': 'alert_triage'})
print('Reset response status:', r.status_code)
data = r.json()
print('Task:', data.get('observation', {}).get('task_context', {}).get('task_id', 'N/A'))
print('Alerts in queue:', len(data.get('observation', {}).get('alert_queue', [])))

# Test scenarios endpoint
r2 = requests.get('http://localhost:8000/scenarios')
scenarios = r2.json()
print(f'Available scenarios: {scenarios.get("count", 0)}')
print('Environment is working correctly!')

## Step 6: Configure Training

In [ ]:
import torch

# Check GPU memory to auto-select model size
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    print(f'GPU: {gpu_name} ({gpu_mem_gb:.1f} GB)')

    if gpu_mem_gb >= 14:
        MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
        STEPS = 200
        K = 4
        print('Using 1.5B model (Full GRPO, K=4)')
    else:
        MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
        STEPS = 100
        K = 2
        print('Using 0.5B model (Compact GRPO, K=2)')
else:
    print('WARNING: No GPU. Training will be very slow. Enable T4 GPU in Runtime settings.')
    MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
    STEPS = 10
    K = 2

print(f'Training config: model={MODEL}, steps={STEPS}, K={K}')

## Step 7: Run GRPO Training

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

# Launch training
cmd = [
    'python', 'train/train_grpo.py',
    '--model', MODEL,
    '--steps', str(STEPS),
    '--task', 'all',
    '--group-size', str(K),
    '--output', 'outputs/grpo_checkpoints'
]

print('Starting GRPO training...')
print(f'Command: {" ".join(cmd)}')
print('=' * 60)

# Run with live output
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print('=' * 60)
print(f'Training finished with exit code: {process.returncode}')

## Step 8: View Training Metrics

In [ ]:
import json

metrics_path = 'outputs/grpo_checkpoints/metrics.json'
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)

    print(f'Total steps recorded: {len(metrics)}')
    print()
    print(f'{"Step":<6} {"Task":<25} {"Scenario":<25} {"Mean":>8} {"Max":>8}')
    print('-' * 80)
    for m in metrics:
        print(f"{m['step']:<6} {m['task_id']:<25} {m['scenario_id']:<25} {m['mean_score']:>8.4f} {m['max_score']:>8.4f}")

    best = max(metrics, key=lambda x: x['max_score'])
    print()
    print(f'BEST STEP: Step {best["step"]} — Max Score: {best["max_score"]} on {best["scenario_id"]}')
else:
    print('No metrics file found yet.')

## Step 9: Plot Reward Curves

In [ ]:
import matplotlib.pyplot as plt
import json

metrics_path = 'outputs/grpo_checkpoints/metrics.json'
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)

    steps = [m['step'] for m in metrics]
    mean_scores = [m['mean_score'] for m in metrics]
    max_scores = [m['max_score'] for m in metrics]

    # Rolling average
    window = 5
    rolling_avg = []
    for i in range(len(mean_scores)):
        start = max(0, i - window + 1)
        rolling_avg.append(sum(mean_scores[start:i+1]) / (i - start + 1))

    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(steps, mean_scores, alpha=0.4, color='#00bfff', linewidth=1, label='Mean Reward')
    ax.plot(steps, max_scores, alpha=0.4, color='#ff6b6b', linewidth=1, label='Max Reward')
    ax.plot(steps, rolling_avg, color='#00ff88', linewidth=2.5, label=f'Rolling Avg ({window} steps)')
    ax.axhline(y=0.40, color='gold', linestyle='--', alpha=0.7, label='Expert Threshold (0.40)')
    ax.set_xlabel('Training Step', fontsize=12)
    ax.set_ylabel('Reward Score (0-1)', fontsize=12)
    ax.set_title('Mini SOC — GRPO Training Reward Curves', fontsize=14, fontweight='bold')
    ax.legend()
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig('outputs/plots/colab_reward_curves.png', dpi=150)
    plt.show()
    print('Chart saved to outputs/plots/colab_reward_curves.png')
else:
    print('No metrics found. Run training first.')

## Step 10: Merge LoRA Weights into Final Model

In [ ]:
print('Merging LoRA adapter into base model...')
result = subprocess.run(
    ['python', 'train/merge_lora.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode == 0:
    print('Merge complete! Expert model saved to outputs/merged_model/')
else:
    print('Merge error:', result.stderr)

## Step 11: Final Benchmark — Base Model vs. RL Expert

In [ ]:
import requests, time, json

SCENARIOS = [
    ('brute_force_ssh_001', '47 failed SSH logins from 185.220.101.47'),
    ('phishing_lateral_001', 'Phishing email opened, lateral movement detected'),
    ('ransomware_001', 'Mass file encryption process detected on BACKUP-SRV-01'),
    ('insider_threat_001', 'Large data upload to personal cloud storage by finance user'),
    ('ghost_apt_001', 'WMIC lateral movement from 10.0.0.5 spawning powershell on WS-02'),
]

def run_benchmark(model_label):
    results = []
    for sid, desc in SCENARIOS:
        start = time.time()
        try:
            r = requests.post('http://localhost:8000/inference',
                              json={'description': desc, 'alert_id': 'TEST'},
                              timeout=60)
            output = r.json().get('response', '')
            is_json = '{' in output and '}' in output
            elapsed = round(time.time() - start, 1)
            results.append({'scenario': sid, 'json_valid': is_json, 'time': elapsed, 'output': output[:80]})
        except Exception as e:
            results.append({'scenario': sid, 'json_valid': False, 'time': 0, 'output': str(e)})
    return results

print('Running benchmark on RL Expert Model...')
expert_results = run_benchmark('RL Expert')

print()
print('=' * 70)
print('BENCHMARK RESULTS — RL Expert Model')
print('=' * 70)
print(f'{"Scenario":<30} {"JSON Valid":<12} {"Time (s)":<10}')
print('-' * 70)
passed = 0
for r in expert_results:
    status = 'YES' if r['json_valid'] else 'NO'
    if r['json_valid']: passed += 1
    print(f"{r['scenario']:<30} {status:<12} {r['time']:<10}")

print('=' * 70)
print(f'PASS RATE: {passed}/{len(SCENARIOS)} ({100*passed//len(SCENARIOS)}%)')

## Step 12: Save Trained Model to Google Drive (Optional)

In [ ]:
# Run this cell only if you want to save the model to Google Drive
SAVE_TO_DRIVE = False  # Set to True to save

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.system('cp -r outputs/merged_model /content/drive/MyDrive/mini-soc-expert-model')
    os.system('cp outputs/grpo_checkpoints/metrics.json /content/drive/MyDrive/mini-soc-metrics.json')
    print('Model saved to Google Drive!')
else:
    print('Skipping Drive save. Set SAVE_TO_DRIVE = True to enable.')

---
## Done!

Your Mini SOC RL agent has been trained and verified.

| Item | Location |
| :--- | :--- |
| Expert Model Weights | `outputs/merged_model/` |
| Training Metrics | `outputs/grpo_checkpoints/metrics.json` |
| Reward Plot | `outputs/plots/colab_reward_curves.png` |
| LoRA Checkpoint | `outputs/grpo_checkpoints/best/` |